# Capabilities Overview
**`plant_growth_monitor/capabilities/`**

This notebook gives you a complete map of the capabilities module — what exists,  
how to choose the right tool, and how capabilities chain together into use cases.

---

## Module map

```
capabilities/
├── analysis/
│   ├── colour_analyser.py      → HSV health index, green/yellow/brown ratios      → nb 01
│   ├── texture_analyser.py     → LBP + Gabor disease surface texture              → nb 02
│   ├── coverage_estimator.py   → canopy % over time, series plots                → nb 03
│   └── anomaly_detector.py     → DINOv2 unsupervised anomaly detection           → nb 09
│
├── segmentation/
│   ├── hsv_segmentor.py        → classical colour masking, 8 profiles, <10ms     → nb 04
│   └── sam2_segmentor.py       → SAM2 precise masks, box/point/auto prompt       → nb 04
│
├── detection/
│   ├── grounding_dino.py       → open-vocab bounding boxes (best accuracy)       → nb 05
│   ├── owlvit_detector.py      → open-vocab detection (CPU-friendly)             → nb 05
│   └── yolo_detector.py        → ultra-fast COCO detection (<50ms)               → nb 05
│
├── classification/
│   ├── clip_classifier.py      → zero-shot labelling, multi-question             → nb 06
│   └── vlm_classifier.py       → Gemini reasoning + structured schema            → nb 06
│
├── depth/
│   ├── depth_estimator.py      → Depth Anything V2, height estimation            → nb 07
│   └── midas_depth.py          → MiDaS fallback, identical interface             → nb 07
│
└── temporal/
    ├── growth_tracker.py       → time-series log, alerts, plot                   → nb 08
    └── change_detector.py      → frame-to-frame change, diff maps                → nb 08
```

## Decision guide — which capability for which task?

| Question you want to answer | Best capability | Backup |
|-----------------------------|----------------|--------|
| Is there a sprout? | `CLIPClassifier` | `ColourAnalyser.health_index` |
| Where exactly is the plant? | `OWLViTDetector` | `GroundingDINODetector` |
| What is the precise plant shape? | `SAM2Segmentor` (from box) | `HSVSegmentor` |
| How tall is the plant? | `DepthEstimator.estimate_height()` | `MiDaSDepth` |
| Is the plant healthy? | `ColourAnalyser.health_index` | `CLIPClassifier` |
| Is there disease / spots? | `AnomalyDetector` | `TextureAnalyser.anomaly_score` |
| What growth stage is it? | `CLIPClassifier` (stage labels) | `VLMClassifier.classify()` |
| How much canopy coverage? | `CoverageEstimator` | `HSVSegmentor.coverage_pct` |
| Did the plant change overnight? | `ChangeDetector` | `CoverageEstimator` diff |
| Is it growing as expected? | `GrowthTracker.growth_rate()` | manual |
| What does the grower need to do? | `VLMClassifier.analyse_structured()` | — |

## Setup

In [ ]:
import sys, os, tempfile
sys.path.insert(0, os.path.abspath('../..'))

import cv2
import numpy as np
import matplotlib.pyplot as plt

from sprout_detection.utils.image_gen import make_sprout_image, make_bare_soil_image

tmp = tempfile.mkdtemp()
img_path = os.path.join(tmp, 'plant.png')
cv2.imwrite(img_path, make_sprout_image(seed=42))

print('Test image ready:', img_path)

## Tier 1 — free, CPU, no download (<10ms total)

In [ ]:
from capabilities.analysis.colour_analyser   import ColourAnalyser
from capabilities.analysis.coverage_estimator import CoverageEstimator
from capabilities.analysis.texture_analyser  import TextureAnalyser
from capabilities.segmentation.hsv_segmentor import HSVSegmentor
from capabilities.temporal.change_detector   import ChangeDetector

# Every call is independent — import only what you need
colour   = ColourAnalyser().analyse(img_path)
coverage = CoverageEstimator().estimate(img_path)
texture  = TextureAnalyser().analyse(img_path)
seg      = HSVSegmentor().segment(img_path, profile='green_plant')

print('Tier 1 results (no models, no downloads):')
print(f'  ColourAnalyser     : health_index={colour.health_index:.3f}  green={colour.green_ratio:.3f}')
print(f'  CoverageEstimator  : coverage={coverage.green_coverage_pct:.2f}%  health_ratio={coverage.health_ratio:.3f}')
print(f'  TextureAnalyser    : class="{texture.texture_class}"  anomaly={texture.anomaly_score:.3f}')
print(f'  HSVSegmentor       : coverage={seg.coverage_pct:.2f}%  regions={seg.component_count}')

total_ms = colour.duration_ms + coverage.duration_ms + texture.duration_ms + seg.duration_ms
print(f'\nTotal time: {total_ms:.1f} ms')

## Tier 2 — model download, CPU-friendly

In [ ]:
# These download models on first call
from capabilities.classification.clip_classifier import CLIPClassifier
from capabilities.detection.owlvit_detector      import OWLViTDetector
from capabilities.depth.depth_estimator          import DepthEstimator

clf      = CLIPClassifier()    # ~350 MB ViT-B/32
detector = OWLViTDetector()    # ~200 MB
depth    = DepthEstimator()    # ~100 MB Depth Anything V2 Small

clf_result  = clf.classify(img_path,
    labels=['healthy plant', 'stressed plant', 'bare soil', 'diseased plant'])

det_result  = detector.detect(img_path,
    prompts=['a green seedling', 'a plant sprout'])

depth_result = depth.estimate(img_path)

print('Tier 2 results:')
print(f'  CLIPClassifier  : top="{clf_result.top_label}"  conf={clf_result.confidence:.3f}')
print(f'  OWLViT          : {det_result.count} detection(s)  found={det_result.found}')
if det_result.found:
    print(f'                    best="{det_result.best.label}"  conf={det_result.best.confidence:.3f}')
print(f'  DepthEstimator  : range=[{depth_result.depth_min:.3f}, {depth_result.depth_max:.3f}]')

## Complete pipeline — sprout health check

In [ ]:
# A real-world health check pipeline combining multiple capabilities
# This could be the core of a new use case: "Is my sprout healthy and how tall is it?"

from capabilities.segmentation.hsv_segmentor import HSVSegmentor
from capabilities.analysis.colour_analyser   import ColourAnalyser
from capabilities.analysis.texture_analyser  import TextureAnalyser
from capabilities.classification.clip_classifier import CLIPClassifier

def full_health_check(image_path):
    """Run a complete health assessment on a plant image."""
    results = {}
    
    # Step 1: Segment the plant (no model, instant)
    seg = HSVSegmentor().segment(image_path, profile='green_plant')
    results['coverage_pct'] = seg.coverage_pct
    results['plant_regions'] = seg.component_count
    
    # Step 2: Colour health (no model, instant)
    colour = ColourAnalyser().analyse(image_path, mask=seg.mask)
    results['health_index'] = colour.health_index
    results['green_ratio']  = colour.green_ratio
    results['yellow_ratio'] = colour.yellow_ratio
    results['health_summary'] = colour.summary
    
    # Step 3: Texture (disease check, no model, fast)
    texture = TextureAnalyser().analyse(image_path, mask=seg.mask)
    results['texture_class']  = texture.texture_class
    results['anomaly_score']  = texture.anomaly_score
    
    # Step 4: CLIP classification (~350 MB, loads once)
    clf = CLIPClassifier()
    clf_result = clf.classify(
        image_path,
        labels=['healthy plant', 'mildly stressed', 'severely stressed', 'diseased'],
        positive_indices=[0]
    )
    results['clip_label']    = clf_result.top_label
    results['clip_positive'] = clf_result.binary_decision
    
    return results


report = full_health_check(img_path)

print('=== Full Plant Health Report ===')
for key, value in report.items():
    if isinstance(value, float):
        print(f'  {key:<20}: {value:.3f}')
    else:
        print(f'  {key:<20}: {value}')

## Chaining detection → segmentation → depth (Use Case B prototype)

In [ ]:
# This is the canonical Use Case B pipeline
from capabilities.detection.owlvit_detector  import OWLViTDetector
from capabilities.segmentation.sam2_segmentor import SAM2Segmentor
from capabilities.depth.depth_estimator      import DepthEstimator

def estimate_plant_height(image_path):
    """Full chain: detect → segment → depth → height."""
    
    # Step 1: Locate the plant
    det = OWLViTDetector().detect(image_path, prompts=['a green seedling'])
    if not det.found:
        return {'error': 'No plant detected'}
    
    bbox = det.best.bbox   # (x1, y1, x2, y2)
    
    # Step 2: Get precise mask
    seg = SAM2Segmentor().segment_from_box(image_path, box=bbox)
    
    # Step 3: Estimate depth map
    depth = DepthEstimator().estimate(image_path)
    
    # Step 4: Measure height
    x1, y1, x2, y2 = bbox
    cx = int((x1 + x2) / 2)
    tip_point  = (cx, int(y1))   # top of bounding box
    base_point = (cx, int(y2))   # bottom of bounding box
    
    height = depth.estimate_height(tip_point, base_point)
    
    return {
        'bbox':              bbox,
        'detection_conf':    det.best.confidence,
        'mask_area_ratio':   seg.area_ratio,
        'depth_difference':  height['depth_difference'],
        'tip_depth':         height['tip_depth'],
        'base_depth':        height['base_depth'],
    }

# Show the code — run estimate_plant_height(img_path) with real models loaded
print('Use Case B pipeline:')
print()
print('  result = estimate_plant_height("sprout.jpg")')
print('  # → {"bbox": [...], "depth_difference": 0.23, ...}')
print()
print('Each step is a separate capability class — swap any one without changing the others.')

## Cost / speed summary

In [ ]:
summary = [
    ('ColourAnalyser',     'Free',       'None',       '<10ms',   'CPU',      '01'),
    ('TextureAnalyser',    'Free',       'None',       '<50ms',   'CPU',      '02'),
    ('CoverageEstimator',  'Free',       'None',       '<10ms',   'CPU',      '03'),
    ('HSVSegmentor',       'Free',       'None',       '<10ms',   'CPU',      '04'),
    ('SAM2Segmentor',      'Free',       '38–350 MB',  '3–5s',    'CPU/GPU',  '04'),
    ('OWLViTDetector',     'Free',       '200 MB',     '2–3s',    'CPU',      '05'),
    ('GroundingDINO',      'Free',       '700 MB',     '5–8s',    'GPU rec.', '05'),
    ('YOLODetector',       'Free',       '6 MB',       '<50ms',   'CPU',      '05'),
    ('CLIPClassifier',     'Free',       '350–900 MB', '1–3s',    'CPU',      '06'),
    ('VLMClassifier',      '~$0.001/img','None (API)', '1–2s',    'Cloud',    '06'),
    ('DepthEstimator',     'Free',       '100–800 MB', '5–30s',   'CPU/GPU',  '07'),
    ('MiDaSDepth',         'Free',       '400 MB',     '8–30s',   'CPU/GPU',  '07'),
    ('GrowthTracker',      'Free',       'None',       'Instant', 'CPU',      '08'),
    ('ChangeDetector',     'Free',       'None',       '<20ms',   'CPU',      '08'),
    ('AnomalyDetector',    'Free',       '90–1100 MB', '2–30s',   'CPU/GPU',  '09'),
]

print(f'{"Class":<22} {"Cost":<14} {"Model":<12} {"Speed CPU":<12} {"Hardware":<10} {"Notebook"}')
print('─' * 85)
for row in summary:
    print(f'{row[0]:<22} {row[1]:<14} {row[2]:<12} {row[3]:<12} {row[4]:<10} nb {row[5]}')